# Adaptive Moving Average Forecasting Demo

This demo investigates whether dynamically adjusting moving average window sizes based on local gradient volatility improves forecasting accuracy over static 3-point moving averages and naive persistence on short Ornstein-Uhlenbeck synthetic time series.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'scikit-learn==1.6.1')

In [ ]:
import os
import json
import urllib.request
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-4b74fb-self-normalized-phase-space-adaptive-mov/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=3) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data_payload = load_data()
print("Loaded dataset successfully with keys:", list(data_payload.keys()))

In [ ]:
# Configuration parameters (minimum scale for demo execution)
N_TRIALS = 10
N_STEPS = 50
THETA = 0.1
MU = 0.0
SIGMA = 0.5
MIN_W = 1
MAX_W = 5

### Core Methods for Forecasting

In [ ]:
def generate_ou_process(n=100, theta=0.1, mu=0.0, sigma=0.5, seed=42):
    np.random.seed(seed)
    x = np.zeros(n)
    for t in range(1, n):
        x[t] = x[t-1] + theta * (mu - x[t-1]) + sigma * np.random.randn()
    return x

def compute_adaptive_ma(series, min_w=1, max_w=5):
    preds = []
    n = len(series)
    for t in range(2, n):
        grad = abs(series[t-1] - series[t-2])
        window = max_w - int(np.clip(grad * 2, 0, max_w - min_w))
        window = max(min_w, min(window, t))
        start = max(0, t - window)
        preds.append(np.mean(series[start:t]))
    return np.array(preds)

def compute_static_ma(series, window=3):
    preds = []
    n = len(series)
    for t in range(2, n):
        start = max(0, t - window)
        preds.append(np.mean(series[start:t]))
    return np.array(preds)

def compute_naive(series):
    preds = []
    n = len(series)
    for t in range(2, n):
        preds.append(series[t-1])
    return np.array(preds)

### Running Trials and Evaluating Models

In [ ]:
mse_adaptive = []
mse_static = []
mse_naive = []

for i in range(N_TRIALS):
    series = generate_ou_process(n=N_STEPS, theta=THETA, mu=MU, sigma=SIGMA, seed=i)
    actuals = series[2:]
    
    pred_adap = compute_adaptive_ma(series, min_w=MIN_W, max_w=MAX_W)
    pred_stat = compute_static_ma(series, window=3)
    pred_naiv = compute_naive(series)
    
    mse_adaptive.append(np.mean((pred_adap - actuals) ** 2))
    mse_static.append(np.mean((pred_stat - actuals) ** 2))
    mse_naive.append(np.mean((pred_naiv - actuals) ** 2))

print(f"Mean MSE (Adaptive MA): {np.mean(mse_adaptive):.4f}")
print(f"Mean MSE (Static MA):   {np.mean(mse_static):.4f}")
print(f"Mean MSE (Naive):       {np.mean(mse_naive):.4f}")

### Results & Visualization

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(len(series)), series, label='OU Series', color='black', alpha=0.6)
plt.plot(range(2, len(series)), pred_adap, label='Adaptive MA', color='blue', linestyle='--')
plt.plot(range(2, len(series)), pred_stat, label='Static MA (w=3)', color='orange', linestyle=':')
plt.plot(range(2, len(series)), pred_naiv, label='Naive Persistence', color='green', alpha=0.5)
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.title('Comparison of Forecast Models on Ornstein-Uhlenbeck Process')
plt.legend()
plt.grid(True)
plt.show()